In [1]:
import os
from pathlib import Path

import plotly.graph_objects as go

from paretodo import InvestigationSpace
from helpers.sampling.sampling_lhs import lhs_sampling
from helpers.problems.hb_butterot_simulator import HimmelblauButterotSimulator

from plotly_theme import register_theme

register_theme()

/Users/ss15425/Documents/00_Git_Repositories/pfgs-optimization/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CONSTRAINT_THRESHOLD = 2000
OBJECTIVE_THRESHOLD = 2000
N_SAMPLES = 20000
SEED = 11
LOCAL_FOLDER = "../.local/"
FOLDER = f"{LOCAL_FOLDER}hb_testsets/"
os.makedirs(FOLDER, exist_ok=True)

x_name = "initial_glucose_concentration"
y_name = "initial_glutamine_concentration"
x_display_name = "Initial Glucose Concentration"
y_display_name = "Initial Glutamine Concentration"

main_output_name = "product_concentration"
constraint_name = "side_product_concentration"
main_display_name = "Titer"
constraint_display_name = "Side Product Concentration"

In [6]:
investigation_space = InvestigationSpace.from_json(Path(f"{LOCAL_FOLDER}hb_is/is.json"))

In [7]:
lhs_samples = lhs_sampling(
    investigation_space=investigation_space,
    n_samples=N_SAMPLES,
    seed=SEED,
)
simulator = HimmelblauButterotSimulator()
experiment_dataset = simulator.run_simulation(lhs_samples, iteration=0)

df = experiment_dataset.to_dataframe()
df.write_csv(f"{FOLDER}testset_hb_full.csv")

df_filtered_obj = df.filter(df["product_concentration"] > OBJECTIVE_THRESHOLD)
df_filtered_obj.write_csv(f"{FOLDER}testset_hb_full_objective.csv")

df_filtered_robust_obj = df.filter(
    (df["product_concentration"] > OBJECTIVE_THRESHOLD) & (df[x_name] > 4)
)
df_filtered_robust_obj.write_csv(f"{FOLDER}testset_hb_robust_objective.csv")

df_filtered_constraint = df.filter(
    df["side_product_concentration"] < CONSTRAINT_THRESHOLD
)
df_filtered_constraint.write_csv(f"{FOLDER}testset_hb_constraint.csv")

df_filtered_const_obj = df.filter(
    (df["product_concentration"] > OBJECTIVE_THRESHOLD)
    & (df["side_product_concentration"] < CONSTRAINT_THRESHOLD)
)
df_filtered_const_obj.write_csv(f"{FOLDER}testset_hb_constraint_objective.csv")

df_filtered_const_obj_rob = df.filter(
    (df["product_concentration"] > OBJECTIVE_THRESHOLD)
    & (df["side_product_concentration"] < CONSTRAINT_THRESHOLD)
    & (df[x_name] > 4)
)
df_filtered_const_obj_rob.write_csv(
    f"{FOLDER}testset_hb_constraint_objective_robust.csv"
)

2026-08-11 08:27:46,603 - root - WARNING - Adding normalization params to a normalized dataset.
2026-08-11 08:27:46,606 - root - WARNING - Adding normalization params to a normalized dataset.


In [8]:
min_conc = df[main_output_name].min()
max_conc = df[main_output_name].max()
x_min = df[x_name].min()
x_max = df[x_name].max()
y_min = df[y_name].min()
y_max = df[y_name].max()

In [9]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df[x_name],
        y=df[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df[main_output_name],
            colorscale="Viridis",
            showscale=True,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])
fig.show()

In [10]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_obj[x_name],
        y=df_filtered_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} > {OBJECTIVE_THRESHOLD} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [11]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_robust_obj[x_name],
        y=df_filtered_robust_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_robust_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"Robust {main_display_name} > {OBJECTIVE_THRESHOLD} Testset",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [12]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_constraint[x_name],
        y=df_filtered_constraint[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_constraint[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [13]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_const_obj[x_name],
        y=df_filtered_const_obj[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_const_obj[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"{main_display_name} > {OBJECTIVE_THRESHOLD} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])

In [14]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df_filtered_const_obj_rob[x_name],
        y=df_filtered_const_obj_rob[y_name],
        mode="markers",
        marker=dict(
            size=5,
            color=df_filtered_const_obj_rob[main_output_name],
            colorscale="Viridis",
            showscale=True,
            cmin=min_conc,
            cmax=max_conc,
        ),
    )
)
fig.update_layout(
    title=f"Robust {main_display_name} > {OBJECTIVE_THRESHOLD} with constraint: {constraint_display_name} < {CONSTRAINT_THRESHOLD}",
    xaxis_title=x_display_name,
    yaxis_title=y_display_name,
    width=800,
    height=600,
)
fig.update_xaxes(range=[x_min, x_max])
fig.update_yaxes(range=[y_min, y_max])